# WP3 - UrbanFormer-Field (flagship)

**Question.** WP2 showed a single pooled `z_geom` cannot carry per-location
geometry. Replace the pooled decoder with **per-query cross-attention to the
building set** plus **axial self-attention over the grid**, so every query cell
reads the buildings that matter to it and the field stays spatially coherent.

**Headline.** UrbanFormer-Field reaches **R2 = 0.8461** (RMSE 0.6192, MAE 0.3722,
rel-L2 0.3553) over fluid cells on the held-out test split, beating the raster
U-Net (0.7194) and the pooled Transformer (0.4421). 1,633,969 parameters.

| model | RMSE | MAE | R2 |
|---|---|---|---|
| WP1 - U-Net (raster) | 0.8217 | 0.4821 | 0.7194 |
| WP2 - Pooled + Fourier/FiLM | 1.1587 | 0.7722 | 0.4421 |
| **WP3 - UrbanFormer-Field** | **0.6192** | **0.3722** | **0.8461** |

**Architecture.** Encoder: permutation-invariant set Transformer over building
tokens `[x_c, y_c, w, l, h]` -> memory. Decoder: `RESIDUAL_DEPTH = 4` residual
blocks of {relative-geometry cross-attention -> axial query self-attention ->
FFN}. The cross-attention bias is anisotropic and streamwise-asymmetric
(`-softplus(g_s) Ds^2 - softplus(g_n) Dn^2 + beta Ds`), encoding wind-aligned
scales and upstream/downstream asymmetry. The model lives in
`urbanformer.models.field`; the loss and dataset are in the package too.

## The axial column-permute fix (architecture revision `uff-axial-fix`)

This is the honest part of the WP3 story. The axial stage attends along grid
rows, then along grid columns. The **column** branch must permute
`(B, Ny, Nx, D) -> (B, Nx, Ny, D)` before reshaping to `(B*Nx, Ny, D)`; reshaping
straight through instead gathers rows a second time, so the model runs
**row attention twice** and never couples cells down a column.

Every UF-F number produced before this fix, including an earlier headline
R2 = 0.8284, came from a model with streamwise coupling only. Same weights,
old vs fixed forward: mean |delta| 0.22, corr 0.96 - the result held but the
mechanism did not, so the fixed model is a full retrain. `field.py` imports the
corrected `AxialSelfAttention` from `urbanformer.models.axial`, whose two
regression tests fail on the row-only variant.

## 1. Setup

In [ ]:
import copy
import math
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from urbanformer.data import TokenFieldDataset, collate_field, grid_query_xy
from urbanformer.models.field import UrbanFormerField
from urbanformer.losses import masked_field_loss, make_radial_bins
from urbanformer.metrics import field_metrics, per_case_rmse

DATA_ROOT  = Path("data/processed")
SPLITS_DIR = Path("data/splits")
CKPT_PATH  = "wp3_uff_axialfix_best.pt"
ARCH_REV   = "uff-axial-fix"

# --- training ---
BATCH_SIZE, NUM_EPOCHS = 8, 80
LR, WEIGHT_DECAY = 1e-3, 1e-4
WARMUP_EPOCHS, GRAD_CLIP, PATIENCE = 5, 1.0, 15
SEED = 0

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (DEVICE.type == "cuda")
torch.manual_seed(SEED)
print("Device:", DEVICE, "| AMP:", USE_AMP)

## 2. Data

`TokenFieldDataset` decodes the **whole 78x78 grid per case** so axial
self-attention and the structural loss act on a coherent field. Each item is
`(tokens, query_xy, qfeats, patches, target_map, fluid)`. Train mode applies
spanwise reflection (exact for +x wind) and periodic translation.

In [ ]:
def load_split(split_file, data_root):
    with open(split_file) as f:
        names = [ln.strip() for ln in f if ln.strip()]
    return [data_root / name for name in names]

train_dirs = load_split(SPLITS_DIR / "train_cases.txt", DATA_ROOT)
val_dirs   = load_split(SPLITS_DIR / "val_cases.txt",   DATA_ROOT)
test_dirs  = load_split(SPLITS_DIR / "test_cases.txt",  DATA_ROOT)
print(f"train/val/test: {len(train_dirs)} / {len(val_dirs)} / {len(test_dirs)}")

pin = (DEVICE.type == "cuda")
train_loader = DataLoader(TokenFieldDataset(train_dirs, train=True), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=pin, collate_fn=collate_field)
val_loader   = DataLoader(TokenFieldDataset(val_dirs, train=False), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=pin, collate_fn=collate_field)

## 3. Model

Parameter count is a fixed fingerprint of the architecture (the column-permute
fix adds no parameters, so this is unchanged from the pre-fix model).

In [ ]:
model = UrbanFormerField().to(DEVICE)
print("params:", f"{sum(p.numel() for p in model.parameters()):,}")   # 1,633,969

# forward-pass shape check on a synthetic 16x16 grid
_Ny = _Nx = 16
_qxy = torch.from_numpy(grid_query_xy(_Ny, _Nx))[None].repeat(2, 1, 1).to(DEVICE)
with torch.no_grad():
    _out = model(torch.rand(2, 10, 5, device=DEVICE),
                 torch.zeros(2, 10, dtype=torch.bool, device=DEVICE),
                 _qxy, torch.rand(2, _Ny*_Nx, 4, device=DEVICE),
                 torch.rand(2, _Ny*_Nx, 81, device=DEVICE), _Ny, _Nx)
print("pred:", tuple(_out.shape), "(expect (2, 256))")

## 4. Training

Loss: masked tail-weighted MSE + finite-difference gradient + radial-PSD spectral
term, all over fluid cells (`urbanformer.losses.masked_field_loss`). The spectral
and gradient terms penalise the missing high-frequency energy directly - they
punish the blur that plain MSE rewards. Schedule: linear warmup then cosine
decay; AdamW; gradient clipping.

In [ ]:
def lr_factor(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    prog = (epoch - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))

rbin = make_radial_bins(78, 78, device=DEVICE)   # radial-PSD bins for the spectral term
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)

best_val, best_weights, patience_counter = float("inf"), None, 0

for epoch in range(NUM_EPOCHS):
    model.train(); train_loss = 0.0
    for tokens, pad, qxy, qf, pa, U, fluid in tqdm(train_loader, desc=f"epoch {epoch + 1}"):
        tokens, pad, qxy, qf, pa, U, fluid = (t.to(DEVICE) for t in (tokens, pad, qxy, qf, pa, U, fluid))
        Ny, Nx = U.shape[1], U.shape[2]
        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            pred = model(tokens, pad, qxy, qf, pa, Ny, Nx).reshape(-1, Ny, Nx)
            loss, _ = masked_field_loss(pred, U, fluid, rbin)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    scheduler.step()

    model.eval(); val_loss = 0.0
    with torch.no_grad():
        for tokens, pad, qxy, qf, pa, U, fluid in val_loader:
            tokens, pad, qxy, qf, pa, U, fluid = (t.to(DEVICE) for t in (tokens, pad, qxy, qf, pa, U, fluid))
            Ny, Nx = U.shape[1], U.shape[2]
            pred = model(tokens, pad, qxy, qf, pa, Ny, Nx).reshape(-1, Ny, Nx)
            val_loss += masked_field_loss(pred, U, fluid, rbin)[0].item()
    val_loss /= len(val_loader)
    print(f"epoch {epoch + 1}/{NUM_EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val:
        best_val, patience_counter = val_loss, 0
        best_weights = copy.deepcopy(model.state_dict())
        torch.save({"model": best_weights, "ARCH_REV": ARCH_REV, "val_loss": best_val}, CKPT_PATH)
        print(f"  -> new best ({val_loss:.4f}); saved {CKPT_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  -> early stop at epoch {epoch + 1}"); break

## 5. Evaluation

UF-F already decodes the full grid, so evaluation is one forward pass per case.
Metrics come from `urbanformer.metrics.field_metrics`, identical to every WP.

In [ ]:
model.load_state_dict(best_weights); model.eval()
preds, targets, masks = [], [], []
with torch.no_grad():
    for tokens, pad, qxy, qf, pa, U, fluid in tqdm(DataLoader(
            TokenFieldDataset(test_dirs, train=False), batch_size=BATCH_SIZE,
            shuffle=False, collate_fn=collate_field), desc="evaluating"):
        tokens, pad, qxy, qf, pa = (t.to(DEVICE) for t in (tokens, pad, qxy, qf, pa))
        Ny, Nx = U.shape[1], U.shape[2]
        pred = model(tokens, pad, qxy, qf, pa, Ny, Nx).reshape(-1, Ny, Nx).cpu()
        preds.append(pred); targets.append(U); masks.append(fluid)

P = torch.cat(preds); T = torch.cat(targets); M = torch.cat(masks)
m = field_metrics(P, T, M)
print(f"Test RMSE {m['RMSE']:.4f} | MAE {m['MAE']:.4f} | R2 {m['R2']:.4f} | rel-L2 {m['relL2']:.4f}")

In [ ]:
rows = [
    ("WP1 - U-Net (raster)",        0.8217, 0.4821, 0.7194),
    ("WP2 - Pooled + Fourier/FiLM", 1.1587, 0.7722, 0.4421),
    ("WP3 - UrbanFormer-Field",     m["RMSE"], m["MAE"], m["R2"]),
]
print(f"{'Model':<32}{'RMSE':>9}{'MAE':>9}{'R2':>9}")
for name, r, mae, r2 in rows:
    print(f"{name:<32}{r:>9.4f}{mae:>9.4f}{r2:>9.4f}")

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

case_rmse = per_case_rmse(P, T, M)
best_idx, worst_idx = int(np.nanargmin(case_rmse)), int(np.nanargmax(case_rmse))


def plot_simulation_u(ax, u_norm, solid, title="", vmin=-1.0, vmax=3.0):
    colors = [(0, 0, 0.5), (0, 0.8, 0.8), (0, 0.25, 0), (0.7, 0.7, 0), (0.4, 0, 0)]
    positions = [0, 0.15, (0 - vmin) / (vmax - vmin), 0.55, 1]
    cmap = mcolors.LinearSegmentedColormap.from_list("custom", list(zip(positions, colors)))
    data = u_norm.copy(); data[solid] = 0
    cax = ax.imshow(data, cmap=cmap, interpolation="bicubic", vmin=vmin, vmax=vmax, origin="upper")
    ax.imshow(np.where(solid, 0, np.nan), cmap="gray", interpolation="nearest", origin="upper", zorder=2)
    ax.set_aspect("equal"); ax.set_title(title); ax.grid(False)
    plt.colorbar(cax, ax=ax, orientation="horizontal", fraction=0.046, pad=0.15, extend="both")


def plot_case(idx, title):
    pred, target, mask = P[idx].numpy(), T[idx].numpy(), M[idx].numpy()
    solid = mask == 0
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{title}  |  RMSE={case_rmse[idx]:.4f}", fontsize=13)
    plot_simulation_u(axes[0], target, solid, title="CFD (ground truth)")
    plot_simulation_u(axes[1], pred,   solid, title="UrbanFormer-Field")
    im = axes[2].imshow(np.abs(pred - target) * mask, cmap="hot", origin="upper")
    axes[2].set_title("|error|"); plt.colorbar(im, ax=axes[2], orientation="horizontal", fraction=0.046, pad=0.15)
    plt.tight_layout(); plt.show()

plot_case(best_idx,  "Best case")
plot_case(worst_idx, "Worst case")